In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
train="/content/train.csv"
test="/content/test.csv"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df_train=pd.read_csv(train)
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2662 entries, 0 to 2661
Columns: 427 entries, id to Group 424
dtypes: float64(1), int64(425), object(1)
memory usage: 8.7+ MB


In [ ]:
df=df_train.copy()
# rdkit lib. is used to for informationchemistry, like it help converting chemical names/structure into data
from rdkit import Chem
from rdkit.Chem import Descriptors

physchem_rows = []
fp_rows = []
for index, rows in df.iterrows():
  mol = Chem.MolFromSmiles(rows['SMILES'])
  mol_weight = Descriptors.MolWt(mol)
  # logp = Descriptors.MolLogP(mol)
  # print(f"Molecular weight: {mol_weight:.2f}")
  df.at[index, 'MolWt'] = mol_weight

In [ ]:
X=df.drop(columns=['SMILES','Tm','id'])
y=df['Tm']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# !pip install catboost

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

model = CatBoostRegressor(iterations=750,
                          learning_rate=0.07,
                          random_state=42,
                          verbose=200)

# Fit the model
model.fit(X_train, y_train)

# Predict and Evaluate
pred = model.predict(X_test)
mae = mean_absolute_error(y_test, pred)

print(f"MAE: {mae}")

0:	learn: 83.1130472	total: 49.2ms	remaining: 36.9s
200:	learn: 46.6673952	total: 446ms	remaining: 1.22s
400:	learn: 39.5428137	total: 860ms	remaining: 749ms
600:	learn: 35.4201218	total: 1.25s	remaining: 309ms
749:	learn: 33.1810963	total: 1.53s	remaining: 0us
MAE: 34.36395332528144


#### Feature Engg.

In [ ]:
df=df_train.copy()

In [ ]:
# !pip install rdkit

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, DataStructs
# =========================
# Config
# =========================
SMILES_COL = "SMILES"
FP_BITS = 2048
FP_RADIUS = 2   # ECFP4
# =========================
# Descriptor functions
# =========================
def get_physchem_descriptors(mol):
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
        "RotB": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "HeavyAtoms": mol.GetNumHeavyAtoms(),
        "FormalCharge": Chem.GetFormalCharge(mol),
    }
def get_morgan_fp(mol, radius=FP_RADIUS, nBits=FP_BITS):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr
# =========================
# Main encoder
# =========================
def encode_smiles(df):
    physchem_rows = []
    fp_rows = []
    for smi in df[SMILES_COL]:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            # handle invalid SMILES
            physchem_rows.append({k: np.nan for k in get_physchem_descriptors(Chem.MolFromSmiles("C")).keys()})
            fp_rows.append(np.zeros(FP_BITS))
            continue
        # descriptors
        physchem = get_physchem_descriptors(mol)
        physchem_rows.append(physchem)
        # fingerprint
        fp = get_morgan_fp(mol)
        fp_rows.append(fp)
    physchem_df = pd.DataFrame(physchem_rows)
    fp_df = pd.DataFrame(fp_rows, columns=[f"ECFP_{i}" for i in range(FP_BITS)])
    # merge
    df_encoded = pd.concat([df.reset_index(drop=True), physchem_df, fp_df], axis=1)

    return df_encoded
# =========================
# Usage
# =========================
df_encoded = encode_smiles(df)

print("Original shape:", df.shape)
print("Encoded shape:", df_encoded.shape)

[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerator
[10:32:35] DEPRECATION WARNING: please use MorganGenerat

Original shape: (2662, 427)
Encoded shape: (2662, 2485)


In [ ]:
X=df_encoded.drop(columns=['id','SMILES','Tm'])
y=df_encoded['Tm']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

#### Model

In [ ]:
from lightgbm import LGBMRegressor

model=LGBMRegressor(n_estimators=750,
                    learning_rate=0.07,
                    random_state=42)
model.fit(X_train, y_train)

pred=model.predict(X_val)
mean_absolute_error(pred, y_val)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005944 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1570
[LightGBM] [Info] Number of data points in the train set: 2129, number of used features: 357
[LightGBM] [Info] Start training from score 277.791617


31.013617760646174

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

model = CatBoostRegressor(iterations=750,
                          learning_rate=0.07,
                          random_state=42,
                          verbose=200)

model.fit(X_train, y_train)


pred = model.predict(X_val)
mae = mean_absolute_error(y_val, pred)

print(f"MAE: {mae}")

0:	learn: 82.5777550	total: 11.1ms	remaining: 8.31s
200:	learn: 37.1063202	total: 1.81s	remaining: 4.95s
400:	learn: 29.7033962	total: 3.65s	remaining: 3.18s
600:	learn: 26.2020257	total: 5.47s	remaining: 1.36s
749:	learn: 24.1253302	total: 6.92s	remaining: 0us
MAE: 29.82523438652029


# Testing

In [ ]:
df_test=pd.read_csv(test)
# df_test.head()

In [ ]:
test_df = encode_smiles(df_test)

[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerator
[10:33:27] DEPRECATION WARNING: please use MorganGenerat

In [ ]:
prediction=model.predict(test_df.drop(columns=['id','SMILES']))

In [ ]:
submission=pd.DataFrame({'id':df_test['id'], 'Tm':prediction})
submission.to_csv("submission.csv", index=False)